# Mat & Comp – RN 6: Entrenar mejor
## Adam, dropout, aumento de datos e inicialización

**Curso:** Redes neuronales (Sección 4) · **Duración sugerida:** 2 sesiones de 50 minutos · **Requisito:** RN 5

### Objetivos de aprendizaje
Al terminar esta lección podrás:
1. Explicar las ideas del **impulso** (momentum) y del optimizador **Adam**, programarlos con numpy y usarlos en PyTorch.
2. Reconocer el **sobreajuste** de una red comparando las curvas de entrenamiento y de prueba.
3. Usar **dropout** y entender por qué la red se comporta distinto con `red.train()` y con `red.eval()`.
4. Aplicar **aumento de datos** con `torchvision.transforms` y medir cuánto mejora la red con pocos datos.
5. Explicar por qué importa la **inicialización** de los pesos y qué pasa si todos empiezan en cero.

## 🔍 ¿Qué tan grande debe ser el paso?

En RN 4 y RN 5 entrenamos siempre con `optim.SGD(..., lr=0.1)`. ¿Por qué 0.1? Porque lo probamos y funcionó. Pero cada red y cada problema piden un paso distinto, y probar a ciegas es lento.

Imagina que bajas una montaña con niebla, sintiendo solo la inclinación bajo tus pies. Si el valle es una **zanja larga y angosta**, la pendiente más fuerte apunta hacia la pared de la zanja, no hacia el fondo del valle.

✏️ *¿Qué le pasa a alguien que siempre da un paso del mismo tamaño en la dirección de mayor pendiente dentro de una zanja angosta? ¿Se te ocurre una forma de caminar más inteligente?*

**Mi respuesta:**

Hoy veremos cuatro trucos que usan quienes entrenan redes de verdad: **mejores optimizadores**, **dropout**, **aumento de datos** e **inicialización** cuidadosa.

## 1. Optimizadores a mano: una zanja angosta

Usaremos una función de dos variables con forma de zanja: muy empinada en la dirección $y$ y casi plana en la dirección $x$. El mínimo está en $(0, 0)$.

$$f(x, y) = \tfrac{1}{2}x^2 + 50\,y^2 \qquad \text{pendientes: } \frac{\partial f}{\partial x} = x, \quad \frac{\partial f}{\partial y} = 100\,y$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def f(p):
    return 0.5 * p[0] ** 2 + 50 * p[1] ** 2

def pendiente(p):
    return np.array([p[0], 100 * p[1]])

inicio = np.array([-8.0, 2.0])

def dibujar_zanja(eje):
    xs, ys = np.meshgrid(np.linspace(-9, 3, 200), np.linspace(-2.5, 2.5, 200))
    eje.contour(xs, ys, 0.5 * xs ** 2 + 50 * ys ** 2, levels=20, cmap="Greys", linewidths=0.6)
    eje.plot(0, 0, "k*", markersize=12)

### Descenso de gradiente (ML 3)

El de siempre: $\;p \leftarrow p - \alpha \cdot \text{pendiente}$.

In [ ]:
def descenso(alfa, pasos=60):
    p = inicio.copy()
    camino = [p.copy()]
    for _ in range(pasos):
        p = p - alfa * pendiente(p)
        camino.append(p.copy())
    return np.array(camino)

camino_gd = descenso(alfa=0.019)
fig, eje = plt.subplots(figsize=(7, 3.2))
dibujar_zanja(eje)
eje.plot(camino_gd[:, 0], camino_gd[:, 1], "o-", markersize=3, label="descenso de gradiente")
eje.legend(); eje.set_title("60 pasos con α = 0.019")
plt.show()
print("Punto final:", camino_gd[-1].round(3), "  f =", round(f(camino_gd[-1]), 4))

El camino **zigzaguea** de pared a pared en la dirección $y$ y avanza muy despacio en la dirección $x$. Y no podemos subir $\alpha$: con $\alpha > 0.02$ el zigzag en $y$ crece y el método **diverge** (pruébalo en *¿Qué pasa si…?*). La dirección empinada obliga a dar pasos diminutos, y en la dirección plana esos pasos casi no avanzan: tras 60 pasos seguimos lejos del mínimo.

### Impulso (momentum)

Idea: en vez de seguir solo la pendiente de **ahora**, seguimos un **promedio** de las pendientes recientes, como una bola pesada que rueda. Los zigzags se cancelan en el promedio, y el avance constante en $x$ se acumula:

$$v \leftarrow \beta\, v + \text{pendiente}, \qquad p \leftarrow p - \alpha\, v \qquad (\beta \approx 0.9)$$

In [ ]:
def impulso(alfa, beta=0.9, pasos=60):
    p = inicio.copy()
    v = np.zeros(2)
    camino = [p.copy()]
    for _ in range(pasos):
        v = beta * v + pendiente(p)
        p = p - alfa * v
        camino.append(p.copy())
    return np.array(camino)

### Adam

**Adam** (2014) es hoy el optimizador más usado. Combina dos ideas:

1. **Impulso:** $m$ es el promedio de las pendientes recientes.
2. **Paso a la medida de cada parámetro:** $s$ es el promedio de los **cuadrados** de las pendientes, es decir, qué tan grande suele ser la pendiente de ese parámetro. Adam divide por $\sqrt{s}$: los parámetros con pendientes enormes dan pasos más cortos y los de pendientes pequeñitas dan pasos más largos.

$$m \leftarrow \beta_1 m + (1-\beta_1)\,g, \qquad s \leftarrow \beta_2 s + (1-\beta_2)\,g^2, \qquad p \leftarrow p - \alpha\,\frac{\hat m}{\sqrt{\hat s} + \varepsilon}$$

donde $g$ es la pendiente, $\beta_1 = 0.9$, $\beta_2 = 0.999$, y $\hat m$, $\hat s$ son $m$ y $s$ con una pequeña corrección para los primeros pasos (cuando los promedios aún están cerca de cero).

In [ ]:
def adam(alfa, b1=0.9, b2=0.999, eps=1e-8, pasos=60):
    p = inicio.copy()
    m = np.zeros(2)
    s = np.zeros(2)
    camino = [p.copy()]
    for t in range(1, pasos + 1):
        g = pendiente(p)
        m = b1 * m + (1 - b1) * g
        s = b2 * s + (1 - b2) * g ** 2
        m_hat = m / (1 - b1 ** t)      # corrección de los primeros pasos
        s_hat = s / (1 - b2 ** t)
        p = p - alfa * m_hat / (np.sqrt(s_hat) + eps)
        camino.append(p.copy())
    return np.array(camino)

caminos = {
    "descenso (α = 0.019)": descenso(0.019),
    "impulso (α = 0.004, β = 0.9)": impulso(0.004),
    "Adam (α = 1)": adam(1.0),
}

fig, eje = plt.subplots(figsize=(8, 3.6))
dibujar_zanja(eje)
for nombre, c in caminos.items():
    eje.plot(c[:, 0], c[:, 1], "o-", markersize=3, label=nombre)
eje.legend(loc="upper right", fontsize=9); eje.set_title("Tres formas de bajar por la zanja (60 pasos)")
plt.show()

for nombre, c in caminos.items():
    print(f"{nombre:30} punto final = {c[-1].round(2)}   f = {f(c[-1]):.3f}")

Con los mismos 60 pasos, el impulso y Adam llegan muy cerca del mínimo, mientras el descenso de gradiente sigue lejos. Fíjate en que Adam da, desde el principio, pasos de tamaño parecido en $x$ y en $y$, aunque las pendientes en $y$ sean 100 veces más grandes: ¡eso es dividir por $\sqrt{s}$! (Cerca del mínimo, Adam y el impulso oscilan un poco alrededor del fondo; en las redes esto casi no importa.)

✏️ *En el primer paso de Adam, $\hat m = g$ y $\hat s = g^2$. ¿Cuánto vale entonces $\hat m / \sqrt{\hat s}$ para cada coordenada? ¿Qué tan largo es el primer paso?*

## 2. Adam en PyTorch

En PyTorch, cambiar de optimizador es cambiar **una línea**. Comparemos, con la red densa de RN 4 y una sola época, SGD con dos pasos distintos contra Adam con su paso típico, $\alpha = 0.001$.

In [ ]:
import torch
from torch import nn, optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import time

entreno = datasets.MNIST("datos", train=True, download=True, transform=transforms.ToTensor())
prueba = datasets.MNIST("datos", train=False, download=True, transform=transforms.ToTensor())
cargador_prueba = DataLoader(prueba, batch_size=1000)
criterio = nn.CrossEntropyLoss()

def entrenar_una_epoca(red, cargador, optimizador):
    red.train()
    suma = 0
    for X, y in cargador:
        perdida = criterio(red(X), y)
        optimizador.zero_grad()
        perdida.backward()
        optimizador.step()
        suma += perdida.item() * len(X)
    return suma / len(cargador.dataset)

def evaluar(red, cargador, desplazamiento=0):
    """Devuelve (exactitud, pérdida promedio)."""
    red.eval()
    correctos, suma = 0, 0
    with torch.no_grad():
        for X, y in cargador:
            salida = red(torch.roll(X, shifts=desplazamiento, dims=3))
            correctos += (salida.argmax(dim=1) == y).sum().item()
            suma += criterio(salida, y).item() * len(X)
    return correctos / len(cargador.dataset), suma / len(cargador.dataset)

def red_densa():
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))

`evaluar` es la función `exactitud` de RN 5, que ahora también devuelve la **pérdida** en prueba (la usaremos en la sección 3).

In [ ]:
opciones = {
    "SGD, α = 0.01": lambda p: optim.SGD(p, lr=0.01),
    "SGD, α = 0.1": lambda p: optim.SGD(p, lr=0.1),
    "Adam, α = 0.001": lambda p: optim.Adam(p, lr=0.001),
}

for nombre, crear in opciones.items():
    torch.manual_seed(0)
    red = red_densa()
    optimizador = crear(red.parameters())
    cargador = DataLoader(entreno, batch_size=64, shuffle=True)
    inicio_t = time.time()
    entrenar_una_epoca(red, cargador, optimizador)
    exa, _ = evaluar(red, cargador_prueba)
    print(f"{nombre:16} exactitud tras 1 época = {exa:.1%}   ({time.time() - inicio_t:.0f} s)")

Adam, con su paso de siempre, le gana a SGD sin que tengamos que buscar el mejor $\alpha$. Por eso es el optimizador **por defecto** de muchísimos proyectos: `optim.Adam(red.parameters(), lr=0.001)`. ¡Pero ojo! El paso de Adam también importa: pruébalo con `lr=0.1` en *¿Qué pasa si…?*.

## 3. Sobreajuste: cuando la red memoriza

En ML 7 vimos que un modelo demasiado flexible puede **memorizar** los datos de entrenamiento en lugar de aprender la regla. Las redes neuronales, con cientos de miles de parámetros, son expertas en memorizar.

Hagamos la prueba: una red **grande** (784 → 512 → 512 → 10, más de 660 000 parámetros) entrenada con solo **1000** imágenes durante 30 épocas. Guardamos la pérdida en entrenamiento y en prueba después de cada época.

In [ ]:
pocas = Subset(entreno, range(1000))           # solo las primeras 1000 imágenes
cargador_pocas_eval = DataLoader(pocas, batch_size=1000)

def red_grande(p=0.0):
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 512), nn.ReLU(), nn.Dropout(p),
        nn.Linear(512, 512), nn.ReLU(), nn.Dropout(p),
        nn.Linear(512, 10),
    )
# nn.Dropout(0.0) no hace nada: por ahora, es una red normal

def entrenar_y_seguir(red, datos, epocas=30, aumentados=None):
    """Entrena con Adam y anota exactitud y pérdida en entrenamiento y prueba en cada época."""
    optimizador = optim.Adam(red.parameters(), lr=0.001)
    cargador = DataLoader(aumentados if aumentados is not None else datos, batch_size=64, shuffle=True)
    historia = []
    for epoca in range(epocas):
        entrenar_una_epoca(red, cargador, optimizador)
        historia.append(evaluar(red, DataLoader(datos, batch_size=1000)) + evaluar(red, cargador_prueba))
    return np.array(historia)    # columnas: exa_entreno, perd_entreno, exa_prueba, perd_prueba

torch.manual_seed(0)
red = red_grande()
print(f"Parámetros: {sum(p.numel() for p in red.parameters()):,}")
historia_sin = entrenar_y_seguir(red, pocas)
print(f"Tras 30 épocas: entrenamiento {historia_sin[-1, 0]:.1%}   prueba {historia_sin[-1, 2]:.1%}")

In [ ]:
def graficar(historias, titulo):
    fig, (e1, e2) = plt.subplots(1, 2, figsize=(11, 3.4))
    for nombre, h in historias.items():
        linea, = e1.plot(range(1, 31), h[:, 1], "--", label=f"{nombre}: entrenamiento")
        e1.plot(range(1, 31), h[:, 3], "-", color=linea.get_color(), label=f"{nombre}: prueba")
        e2.plot(range(1, 31), 100 * h[:, 2], "-", color=linea.get_color(), label=nombre)
    e1.set_xlabel("época"); e1.set_ylabel("pérdida"); e1.set_title("Pérdida"); e1.legend(fontsize=8)
    e2.set_xlabel("época"); e2.set_ylabel("% de aciertos en prueba"); e2.set_title("Exactitud en prueba"); e2.legend(fontsize=8)
    for e in (e1, e2):
        e.grid(alpha=0.3)
    plt.suptitle(titulo)
    plt.tight_layout()
    plt.show()

graficar({"sin dropout": historia_sin}, "Red grande, 1000 imágenes")

La pérdida en **entrenamiento** baja hasta casi cero: la red acierta el 100 % de las 1000 imágenes, ¡las memorizó! Pero la pérdida en **prueba** baja unas pocas épocas y luego **sube**: la red está cada vez más segura de respuestas equivocadas en imágenes nuevas. Esa es la firma del sobreajuste.

✏️ *Según la gráfica, ¿en qué época habría sido mejor detener el entrenamiento?*

## 4. Dropout: apagar neuronas al azar

**Dropout** (2014) es un truco sorprendentemente simple: durante el entrenamiento, en cada lote, **apagamos al azar** una fracción $p$ de las neuronas (las ponemos en cero). Así ninguna neurona puede depender demasiado de otra, y la red se ve obligada a aprender de forma **redundante**, como un equipo en el que cualquiera puede faltar.

Con numpy, para una capa de 10 neuronas y $p = 0.5$:

In [ ]:
rng = np.random.default_rng(1)
activaciones = np.array([0.8, 1.2, 0.5, 2.0, 0.3, 1.1, 0.9, 1.5, 0.7, 1.0])
p = 0.5

mascara = rng.random(10) > p                       # True = la neurona sigue encendida
con_dropout = activaciones * mascara / (1 - p)     # apagar y compensar

print("máscara:     ", mascara.astype(int))
print("antes:       ", activaciones)
print("con dropout: ", con_dropout)

¿Por qué dividir por $1 - p$? Porque si apagamos la mitad de las neuronas, la siguiente capa recibe, en promedio, la mitad de la señal. Al **duplicar** las que quedan, el promedio se mantiene igual.

Al **evaluar** no apagamos nada: queremos que la red use todo lo que sabe. Por eso PyTorch distingue dos modos:

In [ ]:
capa = nn.Dropout(p=0.5)
x = torch.ones(10)

capa.train()          # modo entrenamiento: apaga al azar
torch.manual_seed(0)
print("train():", capa(x))
capa.eval()           # modo evaluación: no hace nada
print("eval(): ", capa(x))

Por eso nuestras funciones llaman `red.train()` antes de entrenar y `red.eval()` antes de evaluar (lo veníamos haciendo desde RN 4: ¡ahora sabes para qué sirve!).

Repitamos el experimento de la sección 3 con `Dropout(0.5)` después de cada capa oculta:

In [ ]:
torch.manual_seed(0)
historia_drop = entrenar_y_seguir(red_grande(p=0.5), pocas)
print(f"Sin dropout: prueba {historia_sin[-1, 2]:.1%}   pérdida en prueba {historia_sin[-1, 3]:.3f}")
print(f"Con dropout: prueba {historia_drop[-1, 2]:.1%}   pérdida en prueba {historia_drop[-1, 3]:.3f}")
graficar({"sin dropout": historia_sin, "con dropout": historia_drop}, "Red grande, 1000 imágenes")

Con dropout la pérdida en prueba sube mucho menos y la exactitud mejora un poco. Dropout **frena** el sobreajuste, pero no hace milagros: con solo 1000 imágenes, la red no tiene suficientes ejemplos. ¿Y si pudiéramos **fabricar** más?

## 5. Aumento de datos: fabricar ejemplos nuevos

Un 3 un poco girado o corrido sigue siendo un 3. Si en cada época le mostramos a la red versiones **ligeramente transformadas** de cada imagen, es como tener muchísimas imágenes distintas. Esto se llama **aumento de datos**.

`torchvision.transforms` trae transformaciones al azar listas. `RandomAffine` gira la imagen hasta 10° y la corre hasta un 10 % de su tamaño (unos 3 píxeles):

In [ ]:
aumento = transforms.Compose([
    transforms.RandomAffine(degrees=10, translate=(0.1, 0.1)),
    transforms.ToTensor(),
])
entreno_aumentado = datasets.MNIST("datos", train=True, download=True, transform=aumento)

torch.manual_seed(0)
fig, ejes = plt.subplots(3, 8, figsize=(12, 4.8))
for fila, i in enumerate([0, 1, 2]):
    ejes[fila, 0].imshow(entreno[i][0][0], cmap="gray_r")
    ejes[fila, 0].set_title("original", fontsize=9)
    for col in range(1, 8):
        ejes[fila, col].imshow(entreno_aumentado[i][0][0], cmap="gray_r")   # ¡cada vez sale distinta!
        ejes[fila, col].set_title(f"versión {col}", fontsize=9)
for eje in ejes.flat:
    eje.axis("off")
plt.suptitle("Cada vez que pedimos la imagen, la transformación al azar es distinta")
plt.tight_layout()
plt.show()

Ahora entrenemos la red convolucional de RN 5 con las mismas **1000 imágenes**, con y sin aumento de datos, y midamos la exactitud con el dígito corrido de 0 a 5 píxeles, como en RN 4 y RN 5. (Tarda unos 20 segundos.)

In [ ]:
def red_conv(p=0.0):
    return nn.Sequential(
        nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(), nn.Dropout(p),
        nn.Linear(32 * 7 * 7, 10),
    )

pocas_aumentadas = Subset(entreno_aumentado, range(1000))   # las mismas 1000, transformadas al azar

resultados = {}
for nombre, datos in [("sin aumento", pocas), ("con aumento", pocas_aumentadas)]:
    torch.manual_seed(0)
    red = red_conv()
    optimizador = optim.Adam(red.parameters(), lr=0.001)
    cargador = DataLoader(datos, batch_size=64, shuffle=True)
    for epoca in range(30):
        entrenar_una_epoca(red, cargador, optimizador)
    resultados[nombre] = [evaluar(red, cargador_prueba, d)[0] for d in range(6)]

print("px   sin aumento   con aumento")
for d in range(6):
    print(f"{d:2}   {resultados['sin aumento'][d]:9.1%}   {resultados['con aumento'][d]:11.1%}")

plt.figure(figsize=(5.5, 3.3))
for nombre, exas in resultados.items():
    plt.plot(range(6), [100 * e for e in exas], "o-", label=nombre)
plt.xlabel("píxeles corridos a la derecha"); plt.ylabel("% de aciertos")
plt.title("Red convolucional con 1000 imágenes")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

Con las **mismas** 1000 imágenes, el aumento de datos mejora la exactitud normal y, sobre todo, la robustez: la red aprendió que un dígito corrido sigue siendo el mismo dígito, porque se lo mostramos muchas veces.

**Importante:** el aumento se usa **solo en el entrenamiento**. Las imágenes de prueba se dejan como son: queremos medir la red en datos reales.

✏️ *¿Qué transformaciones **no** tendría sentido usar con dígitos? Piensa en voltear de izquierda a derecha o girar 180°.*

## 6. Inicialización: dónde empiezan los pesos

Antes de entrenar, los pesos se llenan al azar. ¿Importa **cuánto** al azar? Pasemos 500 datos por una red de 10 capas de 100 neuronas con ReLU, con pesos aleatorios de tres tamaños distintos, y midamos la desviación estándar de las activaciones en cada capa.

In [ ]:
rng = np.random.default_rng(0)
entrada = rng.standard_normal((500, 100))

def desviaciones(escala, capas=10):
    a = entrada
    resultado = []
    for _ in range(capas):
        W = rng.standard_normal((100, 100)) * escala
        a = np.maximum(0, a @ W)            # capa lineal + ReLU
        resultado.append(a.std())
    return resultado

escalas = {"pesos grandes (σ = 1)": 1.0,
           "pesos pequeñitos (σ = 0.01)": 0.01,
           "Kaiming (σ = √(2/100) ≈ 0.14)": np.sqrt(2 / 100)}

plt.figure(figsize=(6.5, 3.6))
for nombre, esc in escalas.items():
    plt.semilogy(range(1, 11), desviaciones(esc), "o-", label=nombre)
plt.xlabel("capa"); plt.ylabel("desviación de las activaciones (escala log)")
plt.title("¿Qué le pasa a la señal al atravesar 10 capas?")
plt.xticks(range(1, 11))
plt.legend(fontsize=9); plt.grid(alpha=0.3)
plt.show()

- Con pesos **grandes**, la señal se multiplica por unas 7 en cada capa y **explota** (¡cerca de $10^8$ en la capa 10!).
- Con pesos **pequeñitos**, se divide por unas 14 en cada capa y **se desvanece** hasta casi cero (recuerda el desvanecimiento del gradiente de RN 3).
- Con la escala de **Kaiming**, $\sigma = \sqrt{2/n}$ (con $n$ = entradas de la capa), la señal se mantiene estable. El 2 compensa que la ReLU apaga la mitad de los valores.

PyTorch ya inicializa `nn.Linear` y `nn.Conv2d` con una regla de esta familia, así que normalmente no tenemos que hacer nada. Pero hay un error que sí vale la pena ver: **empezar con todos los pesos en cero**.

In [ ]:
torch.manual_seed(0)
red_ceros = red_densa()
for capa in red_ceros:
    if isinstance(capa, nn.Linear):
        nn.init.zeros_(capa.weight)
        nn.init.zeros_(capa.bias)

torch.manual_seed(0)
red_normal = red_densa()

for nombre, red in [("todo en cero", red_ceros), ("inicialización de PyTorch", red_normal)]:
    optimizador = optim.Adam(red.parameters(), lr=0.001)
    entrenar_una_epoca(red, DataLoader(entreno, batch_size=64, shuffle=True), optimizador)
    print(f"{nombre:26} exactitud tras 1 época = {evaluar(red, cargador_prueba)[0]:.1%}")

print("\n¿Cambiaron los pesos de la primera capa de la red en ceros?", red_ceros[1].weight.abs().sum().item() != 0)

¡La red en ceros no aprende nada! Con todo en cero, las neuronas ocultas dan 0, y la retropropagación (RN 3) les envía una culpa de 0: sus pesos **nunca** se mueven. Solo aprende el sesgo final, que termina respondiendo siempre el dígito más común. Además, si todas las neuronas empiezan iguales, reciben la misma culpa y siguen siendo iguales para siempre. El azar al inicio **rompe esa simetría**.

## 7. Juntando todo

Apliquemos los trucos a la red convolucional de RN 5 con **todas** las 60 000 imágenes: **Adam**, **dropout** antes de la última capa y **aumento de datos**. Solo 3 épocas. (Tarda uno o dos minutos en CPU.)

In [ ]:
torch.manual_seed(0)
red_final = red_conv(p=0.5)
optimizador = optim.Adam(red_final.parameters(), lr=0.001)
cargador = DataLoader(entreno_aumentado, batch_size=64, shuffle=True)

inicio_t = time.time()
for epoca in range(1, 4):
    entrenar_una_epoca(red_final, cargador, optimizador)
    exa, perd = evaluar(red_final, cargador_prueba)
    print(f"Época {epoca}: exactitud en prueba = {exa:.2%}   ({time.time() - inicio_t:.0f} s)")

robustez = [evaluar(red_final, cargador_prueba, d)[0] for d in range(6)]
print("\nCon el dígito corrido 0, 1, 2, 3, 4, 5 px:", " ".join(f"{e:.1%}" for e in robustez))

Compara con RN 5, donde la misma red con SGD y sin aumento llegó a cerca de 98 % y, con el dígito corrido 3 píxeles, bajaba a cerca de 80 %. Ahora la exactitud normal es parecida o mejor, y la red **aguanta** mucho mejor los desplazamientos.

| Truco | Qué resuelve | En PyTorch |
|---|---|---|
| Adam | elegir el paso; zanjas y pendientes muy distintas | `optim.Adam(red.parameters(), lr=0.001)` |
| Dropout | sobreajuste: neuronas que dependen demasiado unas de otras | `nn.Dropout(0.5)` + `red.train()` / `red.eval()` |
| Aumento de datos | pocos datos; falta de robustez | `transforms.RandomAffine(...)` solo en entrenamiento |
| Inicialización | señales que explotan o se desvanecen; simetría | la de PyTorch por defecto; `nn.init` si hace falta |

## 8. Errores comunes

**Error 1: olvidar `red.eval()` al evaluar.** Si la red tiene dropout y se queda en modo entrenamiento, ¡cada vez que le preguntas por la misma imagen responde algo distinto!

In [ ]:
imagen = prueba[0][0].unsqueeze(0)
red_final.train()             # ¡ups! modo entrenamiento
with torch.no_grad():
    for vez in range(3):
        print(f"Vez {vez + 1}: puntajes = {red_final(imagen)[0, :4].numpy().round(2)} ...")
red_final.eval()              # así debe ser
with torch.no_grad():
    print(f"eval():  puntajes = {red_final(imagen)[0, :4].numpy().round(2)} ...")

**Error 2: aumentar también los datos de prueba.** La prueba debe medir la red con imágenes reales. Usa el `transform` con aumento solo al crear el conjunto de entrenamiento.

**Error 3: usar el paso de SGD con Adam.** Con Adam, `lr=0.1` suele ser demasiado; el punto de partida típico es `lr=0.001`.

**Error 4: transformaciones que cambian la respuesta.** Girar 180° convierte un 6 en un 9, y voltear en espejo un 2 puede parecer un 5. El aumento debe producir imágenes que sigan teniendo la **misma etiqueta**.

**Error 5: inicializar todo en cero** (sección 6). La red nunca rompe la simetría.

**Error 6: crear el optimizador antes de crear (o volver a crear) la red.** El optimizador guarda una lista de los parámetros. Si vuelves a ejecutar la celda que crea la red pero no la del optimizador, el optimizador sigue moviendo los parámetros de la red **vieja** y la nueva nunca aprende.

## 🤔 ¿Qué pasa si…?

Usa la celda de abajo. Antes de ejecutar, **predice** el resultado.

1. …usas `descenso(alfa=0.021)` en la zanja de la sección 1? ¿Y `adam(0.1)`?
2. …usas `optim.Adam(red.parameters(), lr=0.1)` con la red densa durante una época?
3. …usas `nn.Dropout(p=0.9)` en la red grande de la sección 4? ¿Y `p=0.0`?
4. …usas `transforms.RandomRotation(180)` como aumento? Mira primero algunas imágenes de 6 y de 9.
5. …en la sección 6 usas 30 capas en vez de 10 con la escala de Kaiming? ¿Y con σ = 0.2?

In [ ]:
# Prueba aquí tus ideas

## ✍️ Ejercicios

**Ejercicio 1 (fácil). Impulso a mano.** Para $f(x) = x^2$ (pendiente $2x$), empezando en $x = 4$ con $\alpha = 0.1$ y $\beta = 0.9$, calcula en tu cuaderno $v$ y $x$ durante 3 pasos con la regla del impulso ($v \leftarrow \beta v + \text{pendiente}$, $x \leftarrow x - \alpha v$). Compara con 3 pasos de descenso de gradiente normal. Comprueba con código.

**Ejercicio 2 (fácil). Dropout en promedio.** Con numpy, aplica dropout con $p = 0.5$ (con la compensación $1/(1-p)$) a un vector de 10 000 unos. ¿Qué fracción queda en cero? ¿Cuánto vale el promedio del resultado? Repite con $p = 0.2$ y $p = 0.8$.

**Ejercicio 3 (medio). Buscar el paso de Adam.** Entrena la red densa una época con Adam y `lr` igual a 0.0001, 0.001, 0.01 y 0.1. Haz una gráfica de la exactitud en prueba contra `lr` (usa `plt.semilogx`). ¿Cuál es el mejor? ¿Qué pasa en los extremos?

**Ejercicio 4 (medio). Detenerse a tiempo.** Usando `historia_sin` de la sección 3, encuentra con `np.argmin` la época con la menor pérdida en prueba (columna 3) y su exactitud en prueba (columna 2). Compárala con la exactitud de la última época. A esta idea se le llama **parada temprana** (*early stopping*).

*Nota:* en un proyecto real, esa decisión se toma con un conjunto de **validación** separado del de prueba (ML 7), para no hacer trampa.

**Ejercicio 5 (avanzado). ¿Qué ayuda más con pocos datos?** Con la red convolucional y las 1000 imágenes de la sección 5, entrena 30 épocas las cuatro combinaciones: sin nada, solo dropout (`red_conv(p=0.5)`), solo aumento, y dropout más aumento. Haz una tabla con la exactitud en prueba sin correr y con 3 píxeles corridos. ¿Qué truco aporta más? ¿Se suman sus efectos?

## 🏆 Reto opcional: ¡a por el 99 %!

Con todo lo de hoy, intenta pasar del **99 %** de exactitud en el conjunto de prueba de MNIST con una red convolucional. Ideas:

- Más filtros (32 y 64) o un tercer bloque convolucional (ejercicio 5 de RN 5).
- Una capa oculta `nn.Linear(..., 128)` con dropout antes de la salida.
- Más épocas (5 a 10) y, si puedes, la GPU de Colab: mueve la red y cada lote con `.to("cuda")`.
- Investiga `nn.BatchNorm2d`, otra técnica muy usada para entrenar mejor, y agrégala después de cada convolución.

Anota cada intento en una tabla (cambio, parámetros, épocas, tiempo, exactitud). ¿Qué cambio ayudó más?

## 🎬 Cierre

Hoy aprendiste que:
- El **impulso** promedia las pendientes recientes y suaviza los zigzags; **Adam** además ajusta el paso de cada parámetro dividiendo por el tamaño típico de su pendiente. `optim.Adam(..., lr=0.001)` es un excelente punto de partida.
- Una red grande con pocos datos **memoriza**: la pérdida en entrenamiento baja a cero mientras la de prueba sube.
- **Dropout** apaga neuronas al azar al entrenar (`red.train()`) y no apaga nada al evaluar (`red.eval()`); frena el sobreajuste.
- El **aumento de datos** fabrica ejemplos nuevos con transformaciones que no cambian la etiqueta; con pocos datos es de los trucos más poderosos.
- La **inicialización** importa: pesos muy grandes o muy pequeños hacen explotar o desvanecer la señal, y todo en cero impide aprender.

**Para pensar:** hoy usamos 60 000 imágenes de dígitos, y con 1000 ya costaba. ¿Qué pasa si quieres reconocer, por ejemplo, 5 especies de flores colombianas y solo tienes 200 fotos? Entrenar desde cero no alcanza. En la próxima lección veremos la **transferencia de aprendizaje**: tomar una red que ya aprendió a ver con millones de imágenes y adaptarla a nuestro problema con muy pocos datos. 🌺